# board_sync: camera board vs the engine's expected board (full design, perception)

README "Perception and rules": *detect mismatches between the physical board and the
engine's expected board, and recover*. The engine holds the authoritative game state; the
camera gives a `BoardPrediction` (`board_assembly.ipynb`). Before the robot moves, and after
the human moves, compare them:

* **match**: nothing to do.
* **one legal move away**: the human moved; accept it (M5's `infer_move` does this more
  robustly from probabilities).
* **anything else**: a piece was knocked over / misplaced, the robot dropped a piece, or the
  classifier is wrong. Then either ask the human to fix the board or (if the robot can) put
  the piece back itself. Which one is the policy decision left as TODO.

Milestone: full design (beyond the class project M1-M5).

**TODO**
- [ ] `diff_boards`, `explaining_move`, `classify_mismatch`
- [ ] `decide_action`: the recovery policy (ask human vs auto-correct vs re-capture)
- [ ] use per-square confidence: a mismatch on a low-confidence square should first trigger a
      re-capture, not a correction
- [ ] hook into the game loop (owned by the game/engine package) once that exists
- [ ] write the self-check (plan at the bottom)

In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import chess

%run ./labels.ipynb

## Differences

In [ ]:
@dataclass
class SquareDiff:
    square: str
    expected: str    # label on the engine board ("empty" or a FEN letter)
    observed: str    # label seen by the camera


def diff_boards(observed: chess.Board, expected: chess.Board) -> list[SquareDiff]:
    """Squares whose piece differs, in square-index order. Placement only."""
    # TODO: implement (compare board_to_labels of both)
    raise NotImplementedError


def explaining_move(observed: chess.Board, expected: chess.Board) -> chess.Move | None:
    """A legal move from `expected` (with its side to move) whose result matches `observed`."""
    # Design: try every expected.legal_moves on a copy, compare board_fen with observed.
    # Up to ~40 candidates, cheap; python-chess handles castling / en passant / promotion.
    # TODO: implement
    raise NotImplementedError


def classify_mismatch(observed: chess.Board, expected: chess.Board) -> str:
    """"match", "legal_move" (one legal move away), "displaced" (same pieces, other squares),
    "missing" / "extra" (piece count lower / higher than expected) or "changed" (otherwise,
    e.g. a misclassified piece type)."""
    # Design (check in this order):
    # - no diffs -> match; explaining_move found -> legal_move;
    # - fewer / more pieces observed -> missing / extra;
    # - same multiset of pieces on the differing squares -> displaced, else changed.
    # TODO: implement
    raise NotImplementedError

## Recovery policy

In [ ]:
ACTIONS = ("ok", "accept_move", "recapture", "ask_human", "auto_correct")


def decide_action(observed: chess.Board, expected: chess.Board,
                  low_confidence: list[str] | None = None) -> str:
    """Pick one of ACTIONS.

    Inputs: camera board, engine board, and the low-confidence squares of the BoardPrediction.
    """
    # Design (suggested policy to start from):
    #   match -> "ok"; legal_move -> "accept_move" (only when it is the human's turn);
    #   any differing square in low_confidence -> "recapture" (take another frame first);
    #   displaced with 1 piece and the robot just moved -> "auto_correct" (move it back);
    #   everything else -> "ask_human" (show the differing squares, wait for confirmation).
    # Write down in the notes why; log how often each branch fires in real games.
    # TODO: implement
    raise NotImplementedError

## Self-check

In [ ]:
# Self-check plan (write these as asserts once the functions above work), expected = start:
# - start vs start -> "match", no diffs.
# - after 1. e4 -> diffs on ["e2", "e4"], explaining_move == e2e4, "legal_move".
# - knight moved b1 -> b3 (not a legal knight move) -> "displaced".
# - h2 pawn removed -> "missing"; extra black pawn on e4 -> "extra".
# - c1 bishop read as a white pawn -> "changed", diff (expected "B", observed "P").
# - decide_action: one case per branch of your policy.
print("board_sync: self-check not written yet (see plan above)")